# 17 v2 - Two competent, bootstrapped pi0.5 LIBERO members

Run this notebook twice on an **A100/H100 80GB**, first with `MODEL_INDEX=0`, then with `1`.
Unlike the raw-base signal pilot, both members start from the same LIBERO-finetuned checkpoint.
They retain all 40 LIBERO tasks but use independent within-task episode bootstraps and seeds.

The target is a better competence/diversity tradeoff: 6,000 full-model updates at batch 32 and a
conservative `5e-6` learning rate. This is a new v2 experiment with separate Drive paths, manifest,
Hub repositories, and future Supabase labels; it does not overwrite v1.

Full LeRobot checkpoints remain disabled. Every 1,000 global updates, the trainer atomically
replaces one model-only Drive recovery bundle (weights + processors, no optimizer). If Colab
disconnects, restart the runtime and rerun the notebook for the same member: training warm-starts
from that bundle with a fresh optimizer. At completion, the final model is saved to Drive and
validated before Hugging Face upload is attempted.

## 1. Install the exact pinned training stack

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and isolated v2 paths

In [ ]:
import os
from pathlib import Path
import torch
from google.colab import drive
from huggingface_hub import HfApi
from pnp.config import PI05_REPO_ID

drive.mount("/content/drive")

MODEL_INDEX = 0                 # change only this to 1 for the second member
TARGET_STEPS = 6000
BATCH_SIZE = 32                 # observed batch 16 used 40.4/80 GB; drop to 24 if this OOMs
LEARNING_RATE = 5e-6
SCHEDULER_WARMUP_STEPS = 500
SCHEDULER_DECAY_STEPS = 6000
SCHEDULER_DECAY_LR = 5e-7
RECOVERY_EVERY = 1000           # one rolling model-only bundle; never optimizer state
FULL_FINETUNE = True
COMPILE_MODEL = False
WANDB = False

assert MODEL_INDEX in (0, 1)
assert FULL_FINETUNE, "v2 is predeclared as a full-model fine-tune"
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory / 2**30 >= 70, "Use an A100/H100 80GB runtime"

SOURCE_MODEL = PI05_REPO_ID     # lerobot/pi05_libero_finetuned
HF_USER = HfApi().whoami()["name"]
MODEL_REPOS = [f"{HF_USER}/pi05-libero-ft-bootstrap-m0-v2",
               f"{HF_USER}/pi05-libero-ft-bootstrap-m1-v2"]
PERSISTENT_ROOT = Path("/content/drive/MyDrive/pnp_diversity_v2")
MANIFEST_PATH = PERSISTENT_ROOT / "bootstrap_manifest_finetuned_v2.json"
OUTPUT_DIR = Path(f"/content/pi05_diversity_v2/train_m{MODEL_INDEX}")
RECOVERY_DIR = PERSISTENT_ROOT / f"recovery_m{MODEL_INDEX}"
FINAL_DRIVE_DIR = PERSISTENT_ROOT / f"final_model_m{MODEL_INDEX}"

print({"gpu": gpu.name, "gpu_gib": round(gpu.total_memory / 2**30, 1),
       "member": MODEL_INDEX, "source": SOURCE_MODEL,
       "model_repo": MODEL_REPOS[MODEL_INDEX], "output": str(OUTPUT_DIR),
       "recovery": str(RECOVERY_DIR), "final": str(FINAL_DRIVE_DIR),
       "steps": TARGET_STEPS, "batch": BATCH_SIZE, "lr": LEARNING_RATE})

## 3. Build or load the shared v2 bootstrap manifest

Member 1 must load this exact file. The source checkpoint and dataset revisions are pinned inside
the manifest; a stale v1/raw-base manifest fails loudly instead of being reused.

In [ ]:
from pnp.diversity import (bootstrap_manifest_summary,
    build_bootstrap_manifest_from_lerobot, load_bootstrap_manifest,
    save_bootstrap_manifest)

if MANIFEST_PATH.exists():
    manifest = load_bootstrap_manifest(MANIFEST_PATH)
else:
    manifest = build_bootstrap_manifest_from_lerobot(
        source_model=SOURCE_MODEL, seed=260)
    save_bootstrap_manifest(manifest, MANIFEST_PATH)

assert manifest["source_model"] == SOURCE_MODEL, manifest["source_model"]
assert manifest["n_tasks"] == 40 and manifest["n_source_episodes"] == 1693
display(bootstrap_manifest_summary(manifest))
print("manifest:", MANIFEST_PATH)
print("manifest hash:", manifest["manifest_hash"])
print("dataset revision:", manifest["dataset_revision"])
print("source model revision:", manifest["source_model_revision"])

## 4. Download immutable inputs before training

This exposes download failures before GPU training starts. Both LeRobot and Hugging Face reuse the
same local Colab cache; never place the large cache on Drive.

In [ ]:
from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HOME

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_LEROBOT_HOME"] = str(HF_HOME)
snapshot_download(
    repo_id=manifest["dataset_repo_id"], repo_type="dataset",
    revision=manifest["dataset_revision"], max_workers=2)
snapshot_download(
    repo_id=manifest["source_model"],
    revision=manifest["source_model_revision"], max_workers=2)
print("Immutable dataset and source checkpoint are cached in:", HF_HOME)

## 5. Verify Hub destination and launch

The first 20-50 updates are the memory preflight. Watch `nvidia-smi`; batch 32 should stay below
roughly 72 GiB. If it OOMs, restart the runtime and set `BATCH_SIZE=24`.

`--no-checkpoints` is unconditional. Recovery snapshots are model-only and keep exactly one Drive
copy. On a warm restart, the printed `global_start_step` is the saved global step and
`session_steps` is only the remaining work; optimizer/scheduler state starts fresh.

In [ ]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")
assert token, "Grant this notebook access to the HF_TOKEN Colab secret"
os.environ["HF_TOKEN"] = token
api = HfApi(token=token)
assert api.whoami()["name"] == HF_USER
api.create_repo(repo_id=MODEL_REPOS[MODEL_INDEX], repo_type="model", exist_ok=True)
print("Write access verified:", MODEL_REPOS[MODEL_INDEX])

In [ ]:
import subprocess, sys

args = [sys.executable, "-u",
        str(Path(package_dir) / "scripts" / "train_pi05_bootstrap.py"),
        "--manifest", str(MANIFEST_PATH), "--member", str(MODEL_INDEX),
        "--output-dir", str(OUTPUT_DIR),
        "--policy-repo-id", MODEL_REPOS[MODEL_INDEX],
        "--steps", str(TARGET_STEPS), "--batch-size", str(BATCH_SIZE),
        "--learning-rate", str(LEARNING_RATE),
        "--scheduler-warmup-steps", str(SCHEDULER_WARMUP_STEPS),
        "--scheduler-decay-steps", str(SCHEDULER_DECAY_STEPS),
        "--scheduler-decay-lr", str(SCHEDULER_DECAY_LR),
        "--save-freq", str(TARGET_STEPS + 1), "--no-checkpoints",
        "--model-only-recovery-dir", str(RECOVERY_DIR),
        "--model-only-recovery-freq", str(RECOVERY_EVERY),
        "--final-drive-dir", str(FINAL_DRIVE_DIR)]
if not FULL_FINETUNE: args.append("--expert-only")
if COMPILE_MODEL: args.append("--compile-model")
if WANDB: args.append("--wandb")

print("starting v2 member", MODEL_INDEX)
process = subprocess.Popen(
    args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=0)
for character in iter(lambda: process.stdout.read(1), ""):
    print(character, end="", flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f"Training exited with code {return_code}")

## 6. Verify the durable final model

In [ ]:
import json

metadata_path = FINAL_DRIVE_DIR / "final_export.json"
assert metadata_path.is_file(), f"missing final export: {metadata_path}"
final_metadata = json.loads(metadata_path.read_text())
assert final_metadata["member_index"] == MODEL_INDEX
assert final_metadata["training_completed_steps"] == TARGET_STEPS
assert final_metadata["manifest_hash"] == manifest["manifest_hash"]
assert list(FINAL_DRIVE_DIR.glob("*.safetensors")), "missing final model weights"
assert not RECOVERY_DIR.exists(), "completed final export should remove superseded recovery"

remote = api.model_info(MODEL_REPOS[MODEL_INDEX])
print({"member": MODEL_INDEX, "repo": MODEL_REPOS[MODEL_INDEX],
       "remote_revision": remote.sha, "manifest_hash": manifest["manifest_hash"],
       "source_model": manifest["source_model"], "source_revision": manifest["source_model_revision"],
       "steps": TARGET_STEPS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
       "final_drive_dir": str(FINAL_DRIVE_DIR)})
print("For member 1, change only MODEL_INDEX and reuse:", MANIFEST_PATH)